In [27]:
import os, sys, warnings, pandas as pd, psycopg
from dotenv import load_dotenv
sys.path.insert(0, "..")
from collector.embed import clean_for_embedding
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
load_dotenv("../.env")
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True, connect_timeout=10)

LABELS = "../data/labels"
CODES = {"crash", "wont_launch", "performance", "bugs", "lost_saves", "servers", "accounts_drm", "controls_ui",
         "price_value", "monetization", "dlc", "cheaters_toxicity", "multiplayer", "updates_support", "gen_ai",
         "politics", "boring_grind", "difficulty_combat", "story", "tutorial", "worse_than_predecessor",
         "none", "not_english", "graphics_audio"}
PATH = f"{LABELS}/review_dev_to_label.csv"          # switch to review_test_to_label.csv for the test set

In [33]:
lab = pd.read_csv(PATH, encoding="utf-8-sig", dtype={"topics": str, "note": str})
START = 20                                            # 0, 10, 20, ...
for i, r in lab.iloc[START:START + 10].iterrows():
    print(f"[{i}] {r['name']}\n{r.review}\n{'-' * 100}")

[20] Pathfinder: Wrath of the Righteous - Enhanced Edition
In-your-face politically motivated character relationships being presented within the first few hours of gameplay isn't something I can recommend in any writing in any piece of media. It was annoying and unnecessary tokenization in The Elder Scrolls: Online, and it's annoying as well as unnecessary tokenization here.

Writing, ideally, should be subtle. The incident that inspired this review was about as subtle as high-impact dentistry via mechanical application of masonry.
----------------------------------------------------------------------------------------------------
[21] Borderlands 4
Permanent stutter, all graphics settings have the same performance with the same issue, can't play a game like that.
----------------------------------------------------------------------------------------------------
[22] Supermarket Simulator
really sucks how the devs ruined such a good game, i highly advise against buying this game since

In [35]:
answers = {
    0: ('controls_ui', "control issues, failed tutorial because of controls"),
    1: ('monetization', 'about what the subscription covers'),
    2: ('controls_ui', 'game runs but technology not compatible'),
    3: ('controls_ui', 'control complaint'),
    4: ('monetization;dlc;price_value', 'complaints about developer choices surrounding in-game purchases, adding paid cosmetics is not an update complaints'),
    5: ('difficulty_combat;boring_grind', 'general complaints'),
    6: ('boring_grind;controls_ui', 'missing conveniences'),
    7: ('difficulty_combat;performance', 'unfair, poor graphic optimization'),
    8: ('lost_saves', 'lost save file'),
    9: ('difficulty_combat', 'npcs stronger than player'),
    10: ('bugs', 'game is a scam because of bugs hidden in late game'),
    11: ('boring_grind;story;graphics_audio', 'repetitive gameplay, poor writing, poor music'),
    12: ('controls_ui', 'terrible controls, UI scaling issues for specifically steam deck'),
    13: ('controls_ui;price_value;tutorial', 'clunky controls, not worth price, building is hard and confusing'),
    14: ('boring_grind;tutorial;difficulty_combat;controls_ui', 'poor tutorial, boring, work like gameplay, boring combat'),
    15: ('boring_grind;difficulty_combat;multiplayer', 'boring gameplayloop, expensive kits can be lost to free kits, multiplayer gameplay ruins experience'),
    16: ('gen_ai', 'dislikes use of AI'),
    17: ('politics;story', 'complaint about generic political view, poor story, twists, writing'),
    18: ('servers', 'disconnections'),
    19: ('wont_launch', 'crashes at startup'),
    20: ('politics;story', 'dislike of politics present, writing not subtle enough'),
    21: ('performance', 'all graphics settings have performance issue'),
    22: ('updates_support;dlc', 'devs ruined game, content hidden behind dlc'),
    23: ('updates_support', 'lack of updates for extended period'),
    24: ('multiplayer;story;cheaters_toxicity;boring_grind;graphics_audio;servers;worse_than_predecessor', 'poor multiplayer due to toxicity, backwards gameplay since previous version, boring story, ugly graphics'),
    25: ('monetization;controls_ui;worse_than_predecessor', 'repetitive pop up ads, cannot remove waypoint, worst game in series'),
    26: ('servers', 'densyncing issues'),
    27: ('bugs', 'bug in last mission'),
    28: ('crash', 'frequent crashes'),
    29: ('price_value;worse_than_predecessor;boring_grind;updates_support', 'not worth price, previous release has better gameplay, more value, little to do post story, post launch support boring'),
    30: ('accounts_drm;monetization', 'sign in inconvenience, poor odds on rare character'),
    31: ('dlc;bugs;price_value', 'overpriced DLC, hole/map bugs'),
    32: ('multiplayer;boring_grind;servers;monetization;crash;performance', 'high change of disconnecting and crashing, lack of content, unable to easily access multiplayer gamemodes, vast monetization, time compiling shaders'),
    33: ('gen_ai', 'dislike for use of AI'),
    34: ('crash', 'common crashes'),
    35: ('bugs;updates_support', 'game breaking bugs, new changes without proper testing'),
    36: ('lost_saves', 'loss of save progress'),
    37: ('updates_support', 'complaints about announcement decision'),
    38: ('boring_grind', 'tedious work'),
    39: ('bugs;crash;controls_ui', 'game crashes, common bugs, performs poorely on Xbox, poor item descriptions'),
    40: ('graphics_audio;story;boring_grind', 'poor voice acting, poor dialog, boring game'),
    41: ('cheaters_toxicity;updates_support', 'cheating is common, believed to be abandoned'),
}

done = sorted(answers)
bad = {i: c for i in done for c in answers[i][0].split(";") if c not in CODES}
assert not bad, f"unknown codes: {bad}"
assert all(answers[i][0] for i in done), "use 'none' for no topic, never an empty string"
lab.loc[done, "topics"] = [answers[i][0] for i in done]
lab.loc[done, "note"] = [answers[i][1] for i in done]
lab.to_csv(PATH, index=False, encoding="utf-8-sig")
print(f"{lab.topics.notna().sum()} of {len(lab)} labeled")

42 of 42 labeled


In [ ]:
TEST_LABEL, TEST_KEY = f"{LABELS}/review_test_to_label.csv", f"{LABELS}/review_test_key.csv"
SEED, CUTOFF = "20261102", "2026-10-25"
QUOTA = {(False, "20-199"): 34, (False, "200-999"): 33, (False, "1000+"): 33,
         (True, "20-199"): 20, (True, "200-999"): 20, (True, "1000+"): 20}
dev_ids = pd.read_csv(f"{LABELS}/review_dev_key.csv").recommendationid.tolist()

pool = pd.read_sql("""
WITH s AS (
    SELECT r.recommendationid, r.appid, r.voted_up,
           CASE WHEN length(btrim(r.review, E' \\t\\r\\n')) < 200  THEN '20-199'
                WHEN length(btrim(r.review, E' \\t\\r\\n')) < 1000 THEN '200-999'
                ELSE '1000+' END AS length_group
    FROM reviews r
    WHERE NOT r.is_short AND r.created_at < %(cutoff)s::date AND NOT (r.recommendationid = ANY(%(dev)s))
), one_per_game AS (
    SELECT *, row_number() OVER (PARTITION BY voted_up, length_group, appid
                                 ORDER BY md5(recommendationid::text || %(seed)s)) AS nth
    FROM s
)
SELECT recommendationid, appid, voted_up, length_group,
       row_number() OVER (PARTITION BY voted_up, length_group ORDER BY md5(recommendationid::text || %(seed)s)) AS k
FROM one_per_game WHERE nth = 1
""", conn, params={"cutoff": CUTOFF, "dev": dev_ids, "seed": SEED})

test = pd.concat([pool[(pool.voted_up == up) & (pool.length_group == g) & (pool.k <= n)]
                  for (up, g), n in QUOTA.items()])
assert len(test) == sum(QUOTA.values()), test.groupby(["voted_up", "length_group"]).size()

texts = pd.read_sql("""SELECT r.recommendationid, g.name, r.review FROM reviews r JOIN games g USING (appid)
                       WHERE r.recommendationid = ANY(%(ids)s)""", conn, params={"ids": test.recommendationid.tolist()})
test = test.merge(texts, on="recommendationid", validate="one_to_one")

if os.path.exists(TEST_LABEL):
    print("test files already exist; not overwriting")
else:
    test = test.sample(frac=1, random_state=int(SEED))                          # shuffle
    test.sort_values("recommendationid")[["recommendationid", "voted_up", "length_group"]].to_csv(TEST_KEY, index=False)
    (test[["recommendationid", "name", "review"]]
        .assign(review=lambda d: d.review.map(clean_for_embedding), topics=None, note=None)
        .to_csv(TEST_LABEL, index=False, encoding="utf-8-sig"))
    print(f"wrote {len(test)} test reviews")